# Hospital Operations Simulator -- How To Run

**SYNTHETIC DEMONSTRATION DATA ONLY -- not derived from, or usable for, real clinical decision-making.**

This is a reference/README notebook (no executable code) for the two Hospital
Operations notebooks in this folder:

- `Hospital_Operations_Setup` -- run **once** to build the operations schema
- `Hospital_Operations_Realtime_Simulator` -- run to continuously produce
  believable operational events (admissions, transfers, discharges,
  staffing/equipment changes, alerts)

Both notebooks are 100% self-contained (no dependency on this git repo's
filesystem) and connect to the **same Fabric SQL database** as the main
`Healthcare_Data_Generator` notebook.

## 1. Prerequisites

1. Run `Healthcare_Data_Generator` at least once first. Hospital Operations
   reads (and never modifies) its clinical tables: `hospitals`, `departments`,
   `floors`, `rooms`, `beds`, `admissions`, `encounters`,
   `patient_bed_assignments`.
2. Make sure the workspace's Fabric SQL database is reachable -- both
   notebooks auto-detect whether they're running inside Fabric and
   authenticate with a non-interactive AAD token; no manual login needed when
   run as a Fabric notebook/job.

## 2. Run order

1. **`Hospital_Operations_Setup`** (run once, safe to re-run any time):
   - Validates the upstream clinical tables exist and are populated
   - Deploys the `ops_*` schema + `vw_ops_*` snapshot views (idempotent)
   - Generates the operational hierarchy (building -> unit -> room attributes)
   - Seeds initial bed/room state from existing patient placements
   - Generates staffing (shifts, staff, current-shift assignments, coverage)
   - Generates equipment inventory + initial state
   - Seeds discharge-readiness + runs an initial alert evaluation
   - Seeds the `ops_simulation_control` row used by the Realtime Simulator
   - Prints a PASS / WARN / FAIL validation summary
2. **`Hospital_Operations_Realtime_Simulator`**: run after Setup completes
   with an acceptable result. Runs a loop that produces small, bounded
   operational changes every iteration and writes them to the `ops_*` tables.

## 3. Key parameters (top config cell of each notebook)

| Parameter | Default | Purpose |
|---|---|---|
| `SCENARIO_NAME` | `NORMAL_OPERATIONS` | See scenario list below |
| `RANDOM_SEED` | `42` | Deterministic runs when set |
| `UPDATE_INTERVAL_SECONDS` | `10` | Wall-clock seconds between iterations |
| `SPEED_MULTIPLIER` | `60` | Simulated time advances this many x faster than wall time |
| `BATCH_SIZE` | `10` | Max rows touched per event type per iteration |
| `MAX_ITERATIONS` / `MAX_RUNTIME_MINUTES` | `0` (unlimited) | Set either to auto-stop after N iterations/minutes |
| `SELECTED_HOSPITAL_ID` | none (all hospitals) | Scope the simulator to one hospital |
| `RESET_CURRENT_STATE` | `false` | Realtime notebook only -- clears current-state tables (never event history) before starting |
| `RESUME_FROM_CHECKPOINT` | `true` | Resume a crashed/interrupted run instead of starting a new one |

Override any of these via environment variables (`HOSPITAL_OPS_*`, see
`.env.example` in the repo) or by editing the config cell directly.

## 4. Scenarios

Set `SCENARIO_NAME` to one of:

- `NORMAL_OPERATIONS` -- baseline
- `FRIDAY_ED_SURGE` -- higher ED arrivals + boarding
- `WINTER_RESPIRATORY_SURGE` -- respiratory demand + ICU pressure
- `ICU_CAPACITY_CRISIS` -- ICU occupancy pressure + transfer delays
- `STAFFING_SHORTAGE` -- higher absence rate, lower RN coverage
- `EQUIPMENT_FAILURE` -- elevated equipment failure rate
- `DISCHARGE_BOTTLENECK` -- slower discharge-readiness progression

These are configuration profiles (tunable probabilities/thresholds), not
separate code paths.

## 5. Controlling a running simulator (pause / resume / stop)

From any other cell/notebook connected to the same database:

```python
set_requested_state(SINGLE_ENGINE, 'Hospital_Operations_Realtime_Simulator', 'PAUSE')
set_requested_state(SINGLE_ENGINE, 'Hospital_Operations_Realtime_Simulator', 'RUN')
set_requested_state(SINGLE_ENGINE, 'Hospital_Operations_Realtime_Simulator', 'STOP')
```

Interrupting the main-loop cell directly (Fabric "Cancel" / Jupyter
"Interrupt Kernel") also triggers a graceful shutdown -- a final checkpoint
is saved and the run is marked `STOPPED_BY_USER` instead of being left
`RUNNING` forever. `RESUME_FROM_CHECKPOINT=true` (default) picks a stale
`RUNNING` run back up from its last checkpoint on the next start.

## 6. Useful queries after (or during) a run

```python
query_db("SELECT * FROM dbo.ops_simulation_run ORDER BY simulation_run_id DESC")
query_db("SELECT * FROM dbo.vw_ops_active_alerts")
query_db("SELECT * FROM dbo.vw_ops_health_system_snapshot")
query_db("SELECT * FROM dbo.vw_ops_hospital_snapshot")
run_all_checks(SINGLE_ENGINE)  # PASS/WARN/FAIL validation, re-runnable any time
```

## 7. Resetting / starting over

- **Current state only** (keeps event history): set `RESET_CURRENT_STATE = True`
  in the Realtime Simulator's config cell, or call `reset_current_state(SINGLE_ENGINE)`.
- **Full teardown** (drops ONLY `ops_*` / `vw_ops_*` objects -- never touches
  clinical tables): call `drop_schema(SINGLE_ENGINE)` from either notebook,
  then re-run `Hospital_Operations_Setup` from the top.

## 8. Troubleshooting

| Symptom | Likely cause / fix |
|---|---|
| Setup raises "No rows found in `floors`/`rooms`" | Run `Healthcare_Data_Generator` first |
| Realtime notebook raises "No ops_* tables found" | Run `Hospital_Operations_Setup` first |
| Run seems stuck `RUNNING` from a previous session | Next start auto-detects the stale heartbeat and resumes from its last checkpoint (or set `RESUME_FROM_CHECKPOINT = False` to start a fresh run instead) |
| Want to change scenario mid-run | Stop the run (`STOP` via `set_requested_state`), edit `SCENARIO_NAME`, restart |
| Validation shows a WARN on the bed-count reconciliation check | Informational only -- compares `ops_unit` staffed-bed sum to the generator's latest `hospital_department_beds.beds_allocated` snapshot; small drift is expected |